# Lesson 1: HTML, the DOM & CSS Selectors

**Week 5 · Data Engineering Course**

---

Last week every row of data arrived in a tidy JSON response because somebody built an API for it. Most of the web is not like that. Job boards, company directories, price lists, government statistics pages — the data is there on the screen, but there is no endpoint to call.

**Web scraping** is how you turn a page built for human eyes into rows in a table.

This lesson teaches you:
- How a web page is structured, and what the DOM is
- How to parse HTML with **BeautifulSoup**
- The difference between `find`, `find_all`, and `select`
- **CSS selectors** — the single most important scraping skill
- How to find the right selector using your browser's DevTools
- How to extract fields safely when some of them are missing
- How to scrape responsibly and legally

**Install before starting:**
```
pip install beautifulsoup4 lxml
```

In [ ]:
!pip install beautifulsoup4 lxml

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

DATA  = Path('data')
PAGES = DATA / 'pages'

print(f'pandas version: {pd.__version__}')
print(f'bs4 ready — practice pages: {[p.name for p in sorted(PAGES.glob("*.html"))]}')

---

## 1.1 From Web Page to DataFrame

Every scraper, no matter how big, is the same four steps:

```
1. FETCH     get the HTML            requests.get(url)  or  Playwright
2. PARSE     build a searchable tree BeautifulSoup(html)
3. EXTRACT   pull out the fields     soup.select(...)
4. STORE     write rows somewhere    DataFrame -> CSV / PostgreSQL
```

This lesson covers steps 2 and 3 — the parsing and extracting — using HTML files already saved on your computer. Working offline first is deliberate: you can re-run a cell a hundred times while you get a selector right, without sending a hundred requests to someone else's server.

Lesson 2 adds step 1 for static sites, Lesson 3 adds it for JavaScript-driven sites, and Lesson 4 covers the storage formats for step 4.

### When should you scrape at all?

Ask these questions in order, and stop at the first "yes":

| Question | If yes |
|----------|--------|
| Is there an official API? | Use the API. It is faster, stabler, and allowed. |
| Is there a bulk download / CSV export? | Download it. |
| Does the site publish the data elsewhere (data portal, GitHub)? | Use that. |
| None of the above? | Now you scrape. |

Scraping is the **last resort**, not the first tool. A scraper breaks every time the site redesigns its HTML; an API contract rarely changes without warning.

---

## 1.2 The Anatomy of HTML

HTML is a tree of **elements**. An element is written as an opening tag, some content, and a closing tag:

```html
<h2 class="job-title">Data Engineer</h2>
│   │                 │              │
│   │                 │              └─ closing tag
│   │                 └─ content (the text you usually want)
│   └─ attribute: name="value"
└─ tag name
```

Three parts matter to a scraper:

1. **Tag name** — `div`, `p`, `a`, `h2`, `table`, `span`, `article`, `li`
2. **Attributes** — extra information on the element. The ones you will use constantly:
   - `class="job-title"` — a reusable style label. **Many elements can share one class.** This is what you usually select on.
   - `id="results"` — a unique label. **Only one element per page may have a given id.**
   - `href="/jobs/1001"` — the destination of a link
   - `datetime="2026-03-02"` — a machine-readable value hiding behind human text like "2 March 2026"
3. **Nesting** — elements contain other elements, which makes the tree:

```
html
└── body
    └── section id="results"
        ├── article class="job-card"
        │   ├── h2 class="job-title"
        │   │   └── a href="..."  "Data Engineer"
        │   ├── p class="company"  "Paystack"
        │   └── ul class="meta"
        │       ├── li class="location"  "Lagos, Nigeria"
        │       └── li class="salary"    "$45,000 - $60,000"
        └── article class="job-card"
            └── ...
```

That tree is called the **DOM** (Document Object Model). BeautifulSoup gives you a Python object for each node in it.

> **Key insight:** scraping a list of things means finding the **repeating element** — here `article.job-card` — then pulling the same fields out of each one. Find the repeating element first; everything else follows.

In [ ]:
# A tiny HTML document, written as a Python string so we can experiment on it
html_doc = """
<html>
  <body>
    <h1 class="site-title">AfriTech Jobs</h1>
    <section id="results">
      <article class="job-card" data-job-id="1001">
        <h2 class="job-title"><a href="jobs/de-1001.html">Data Engineer</a></h2>
        <p class="company">Paystack</p>
        <p class="location">Lagos, Nigeria</p>
      </article>
      <article class="job-card featured" data-job-id="1002">
        <h2 class="job-title"><a href="jobs/ae-1002.html">Analytics Engineer</a></h2>
        <p class="company">Flutterwave</p>
        <p class="location">Remote</p>
      </article>
    </section>
  </body>
</html>
"""

soup = BeautifulSoup(html_doc, 'html.parser')
print(type(soup))

In [ ]:
# prettify() prints the tree with indentation — useful for seeing the structure
print(soup.prettify())

### Which parser should you use?

`BeautifulSoup(html, parser)` needs a parser. You have three realistic choices:

| Parser | Install | Speed | Handles broken HTML |
|--------|---------|-------|---------------------|
| `'html.parser'` | built into Python | medium | reasonably |
| `'lxml'` | `pip install lxml` | **fastest** | well |
| `'html5lib'` | `pip install html5lib` | slowest | exactly like a browser |

Use `'html.parser'` while learning (nothing to install, no surprises) and `'lxml'` in production pipelines where you parse thousands of pages. If a page is so malformed that both give you nonsense, reach for `'html5lib'`.

Always pass the parser explicitly. If you omit it, BeautifulSoup picks one for you and prints a warning — and your script may behave differently on another machine where a different parser is installed.

In [ ]:
# The shortcut syntax: soup.<tag> returns the FIRST matching element
print(soup.h1)             # the whole element, tags included
print(soup.h1.text)        # just the text inside
print(soup.h1['class'])    # the class attribute — note: a LIST, because class can hold several values

In [ ]:
# .text gives you the text of an element AND everything nested inside it
first_card = soup.article
print(repr(first_card.text))      # includes newlines and indentation from the source
print()
print(repr(first_card.get_text(strip=True)))             # strips each string
print(repr(first_card.get_text(separator=' | ', strip=True)))   # join the pieces yourself

---

## 1.3 `find` and `find_all`

These two methods are the classic BeautifulSoup API:

| Call | Returns |
|------|---------|
| `soup.find('p')` | the **first** `<p>` element, or `None` |
| `soup.find_all('p')` | a **list** of every `<p>` element (empty list if none) |

You can filter by attributes as well as by tag:

```python
soup.find_all('article', class_='job-card')      # class_ has a trailing underscore
soup.find_all('article', attrs={'data-job-id': '1001'})
soup.find_all('a', href=True)                    # elements that HAVE an href at all
```

`class_` is spelled with a trailing underscore because `class` is a reserved word in Python. For attribute names that are not valid Python identifiers — like `data-job-id` — use the `attrs={...}` dictionary.

In [ ]:
# find_all returns a list you can loop over
cards = soup.find_all('article', class_='job-card')
print(f'Found {len(cards)} cards\n')

for card in cards:
    title   = card.find('h2', class_='job-title').text
    company = card.find('p', class_='company').text
    print(f'{title:<22} {company}')

In [ ]:
# Pulling attributes out: element['attr'] raises KeyError if missing, element.get('attr') returns None
link = soup.find('a')

print(link['href'])              # 'jobs/de-1001.html'
print(link.get('href'))          # same thing, but safe
print(link.get('target'))        # None — the attribute is not there
print(soup.article.get('data-job-id'))

In [ ]:
# Careful: class_='job-card' matches an element whose class LIST CONTAINS 'job-card'
# The second card has class="job-card featured" — so it matches both of these:
print(len(soup.find_all('article', class_='job-card')))   # 2
print(len(soup.find_all('article', class_='featured')))   # 1

# This is almost always what you want. Multi-valued classes are everywhere in real HTML.
print(soup.find_all('article')[1]['class'])               # ['job-card', 'featured']

---

## 1.4 CSS Selectors — the Skill That Matters

`find_all` works, but it gets verbose fast. **CSS selectors** express the same search in one compact string, and they are the language your browser's DevTools speaks — so you can copy a selector straight out of the browser into your Python.

| Call | Returns |
|------|---------|
| `soup.select_one('.company')` | the first match, or `None` |
| `soup.select('.company')` | a list of all matches |

### The selector syntax you need

| Selector | Meaning | Example match |
|----------|---------|---------------|
| `h2` | every `<h2>` element | `<h2>` |
| `.company` | every element with **class** `company` | `<p class="company">` |
| `#results` | the element with **id** `results` | `<section id="results">` |
| `article.job-card` | `<article>` elements that also have class `job-card` | tag + class together |
| `.job-card .company` | a `.company` **anywhere inside** a `.job-card` (descendant) | any depth |
| `.job-card > p` | a `<p>` that is a **direct child** of `.job-card` | one level only |
| `a[href]` | `<a>` elements that have an `href` attribute | — |
| `[data-job-id="1001"]` | exact attribute value | — |
| `a[href$=".html"]` | `href` **ends with** `.html` (`^=` starts with, `*=` contains) | — |
| `li:nth-of-type(2)` | the second `<li>` among its siblings | — |
| `.job-title a` | the link inside the title | — |
| `h2, h3` | matches `h2` **or** `h3` (comma = or) | — |

The two you will use 90% of the time are **`.class`** and **`descendant` (a space)**. Learn those two properly and the rest is lookup.

> **Remember the space.** `.job-card .company` (with a space) means "a `.company` inside a `.job-card`". `.job-card.company` (no space) means "one element that has *both* classes" — a completely different query.

In [ ]:
# The same extraction as before, with selectors
for card in soup.select('article.job-card'):
    title   = card.select_one('.job-title a').text
    company = card.select_one('.company').text
    href    = card.select_one('.job-title a')['href']
    print(f'{title:<22} {company:<14} {href}')

In [ ]:
# More selector examples on the same document
print('by id:              ', soup.select_one('#results').name)
print('tag + class:        ', len(soup.select('article.job-card')))
print('descendant:         ', [p.text for p in soup.select('.job-card .company')])
print('direct child:       ', [p.text for p in soup.select('.job-card > p')])
print('attribute exists:   ', len(soup.select('a[href]')))
print('attribute equals:   ', soup.select_one('[data-job-id="1002"]').select_one('.company').text)
print('attribute endswith: ', [a["href"] for a in soup.select('a[href$=".html"]')])
print('nth-of-type:        ', soup.select_one('.job-card p:nth-of-type(2)').text)
print('two classes:        ', len(soup.select('.job-card.featured')))

In [ ]:
# select_one returns None when nothing matches — it does NOT raise
missing = soup.select_one('.salary')
print(missing)                    # None

# ...which means this line would crash with AttributeError:
# print(missing.text)

# select returns an empty list, which is safe to loop over
print(soup.select('.salary'))     # []
for s in soup.select('.salary'):
    print('never printed')

---

## 1.5 Walking the Tree

Sometimes the element you want has no useful class of its own, and the only way to describe it is by its position relative to something that does. BeautifulSoup lets you move around the tree:

| Attribute | Meaning |
|-----------|---------|
| `node.parent` | the element that contains this one |
| `node.find_parent('article')` | the nearest ancestor matching a filter |
| `node.children` | direct children (an iterator) |
| `node.find_all('li')` | all `<li>` descendants, at any depth |
| `node.next_sibling` / `previous_sibling` | the node beside this one — **often whitespace text!** |
| `node.find_next_sibling('p')` | the next sibling matching a filter (skips whitespace) |

Use `find_next_sibling()` rather than `next_sibling`: in indented HTML, the raw sibling is usually the newline-and-spaces string between the two tags.

In [ ]:
company = soup.select_one('.company')

print('this element:   ', company.name, '|', company.text)
print('parent:         ', company.parent.name, company.parent.get('class'))
print('find_parent:    ', company.find_parent('article').get('data-job-id'))
print()
print('raw next_sibling:      ', repr(company.next_sibling))         # whitespace!
print('find_next_sibling:     ', company.find_next_sibling('p').text)
print()
# Going from a value back up to its row is a common pattern:
link = soup.select_one('a[href="jobs/ae-1002.html"]')
print('job id for that link:  ', link.find_parent('article')['data-job-id'])

---

## 1.6 Finding Selectors in Your Browser

You will not guess selectors — you will read them off the page. Here is the workflow, in Chrome or Edge:

1. Open the page you want to scrape.
2. **Right-click** the piece of data you want → **Inspect**. DevTools opens with that element highlighted in the Elements panel.
3. Look at the highlighted line. Note its tag, its `class`, and any `id`.
4. Scroll **up** the tree until you find the element that wraps one whole record — one job, one product, one row. That is your repeating element. Note its selector (e.g. `article.job-card`).
5. Test your guess before writing any Python. Press **Ctrl+F** inside the Elements panel and type a selector — DevTools shows how many elements match. `article.job-card` showing "1 of 25" means you found 25 records.
6. Also test in the **Console** tab:
   ```javascript
   document.querySelectorAll('article.job-card').length        // 25
   document.querySelectorAll('article.job-card')[0].innerText  // check the content
   ```
   `querySelectorAll` takes exactly the same selector syntax as `soup.select()`. If it works in the console, it works in BeautifulSoup.

### Two warnings about DevTools

- **"Copy selector" gives you brittle selectors.** Right-click → Copy → Copy selector produces something like `#results > div:nth-child(3) > article > h2`. It works today and breaks the moment the site adds one element. Write selectors by hand using classes that describe meaning (`.job-title`, `.company`), not position.
- **The Elements panel shows the page *after* JavaScript has run.** `requests` only gives you what the server sent. If a selector works in DevTools but finds nothing in Python, you have found a JavaScript-rendered page — that is Lesson 3. To see what `requests` sees, use **View Page Source** (Ctrl+U) instead of Inspect.

---

## 1.7 A Real Practice Page

`data/pages/job_board.html` is a saved job board with ten postings spread over two pages. It has the same quirks you meet in the wild:

- stray whitespace and inconsistent casing in some fields
- a `salary` field that is **missing** on some cards
- human-readable dates (`2 March 2026`) with the machine-readable value hidden in a `datetime` attribute
- a variable number of tags per posting

Open the file in your browser and in a text editor before you run the next cells. You should always look at the HTML before you write a selector for it.

In [ ]:
# Read a local HTML file exactly as you would read any text file
html = (PAGES / 'job_board.html').read_text(encoding='utf-8')
print(f'{len(html):,} characters of HTML')

soup = BeautifulSoup(html, 'html.parser')
print(f'Page title: {soup.title.text}')

cards = soup.select('article.job-card')
print(f'Job cards found: {len(cards)}')

In [ ]:
# ALWAYS look at one record before you write the loop
print(cards[0].prettify())

In [ ]:
# Extract every field from that one card
card = cards[0]

print('job_id:  ', card['data-job-id'])
print('title:   ', card.select_one('.job-title a').get_text(strip=True))
print('url:     ', card.select_one('.job-title a')['href'])
print('company: ', card.select_one('.company').get_text(strip=True))
print('location:', card.select_one('.location').get_text(strip=True))
print('type:    ', card.select_one('.job-type').get_text(strip=True))
print('salary:  ', card.select_one('.salary').get_text(strip=True))
print('posted:  ', card.select_one('.posted time')['datetime'])     # the attribute, not the text
print('tags:    ', [t.get_text(strip=True) for t in card.select('.tag')])

### The problem with that code

Run the same lines against the **second** card and `.salary` returns `None`, so `.get_text()` raises:

```
AttributeError: 'NoneType' object has no attribute 'get_text'
```

Missing fields are not an edge case in scraping — they are the normal case. One malformed record must not kill a run that has already cost you 200 requests. So never chain straight off `select_one`. Write one small helper and use it everywhere.

In [ ]:
# Reproduce the failure on purpose, so you recognise the error message later
card2 = cards[1]
print('salary element:', card2.select_one('.salary'))      # None

try:
    card2.select_one('.salary').get_text(strip=True)
except AttributeError as e:
    print(f'AttributeError: {e}')

In [ ]:
def text_at(node, selector, default=None):
    '''Return the stripped text of the first match for `selector` inside `node`.

    Returns `default` when the selector matches nothing, so a missing
    field never crashes the scrape.
    '''
    found = node.select_one(selector)
    return found.get_text(strip=True) if found else default


def attr_at(node, selector, attribute, default=None):
    '''Return one attribute of the first match for `selector`, or `default`.'''
    found = node.select_one(selector)
    if found is None:
        return default
    return found.get(attribute, default)


# Safe on the card that has no salary
print('salary (card 1):', text_at(cards[0], '.salary'))
print('salary (card 2):', text_at(cards[1], '.salary'))          # None, no crash
print('salary (card 2):', text_at(cards[1], '.salary', 'not stated'))

In [ ]:
def parse_card(card):
    '''Turn one article.job-card element into a dict — one row of our dataset.'''
    return {
        'job_id':   card.get('data-job-id'),
        'title':    text_at(card, '.job-title a'),
        'company':  text_at(card, '.company'),
        'location': text_at(card, '.location'),
        'job_type': text_at(card, '.job-type'),
        'salary':   text_at(card, '.salary'),
        'posted':   attr_at(card, '.posted time', 'datetime'),
        'tags':     ', '.join(t.get_text(strip=True) for t in card.select('.tag')),
        'url':      attr_at(card, '.job-title a', 'href'),
    }

parse_card(cards[1])

In [ ]:
# The whole page in three lines: parse every card into a DataFrame
rows = [parse_card(c) for c in cards]
jobs = pd.DataFrame(rows)

print(jobs.shape)
jobs

---

## 1.8 Scraped Data Is Dirty Data

Look closely at row 1 above. `title` is `analytics engineer` and `company` is `Flutterwave` — in the HTML those fields were wrapped in stray spaces, and `get_text(strip=True)` saved you there. But casing, missing values, and string-typed numbers and dates are still yours to fix.

Everything you learned in Week 3 applies now. Scraping is just a new *source*; the cleaning step is the same.

In [ ]:
clean = jobs.copy()

# 1. Types: ids are integers, dates are dates
clean['job_id'] = clean['job_id'].astype(int)
clean['posted'] = pd.to_datetime(clean['posted'])

# 2. Casing: title-case the job titles so 'analytics engineer' matches 'Analytics Engineer'
clean['title'] = clean['title'].str.title()

# 3. A genuinely missing salary stays missing — do not invent a number.
#    Record the fact instead, so you can filter on it later.
clean['salary_stated'] = clean['salary'].notna()

# 4. Derive useful columns from the text you scraped
clean['is_remote']  = clean['location'].str.lower().eq('remote')
clean['country']    = clean['location'].str.split(',').str[-1].str.strip()
clean.loc[clean['is_remote'], 'country'] = None
clean['seniority']  = 'mid'
clean.loc[clean['title'].str.contains('Senior|Lead|Head', case=False), 'seniority'] = 'senior'
clean.loc[clean['title'].str.contains('Junior|Intern|Graduate', case=False), 'seniority'] = 'junior'

clean[['job_id', 'title', 'company', 'country', 'is_remote', 'seniority', 'salary_stated', 'posted']]

In [ ]:
# Sanity-check the result the way you would check any ingested dataset
print('dtypes:')
print(clean.dtypes)
print()
print('missing values per column:')
print(clean.isna().sum())
print()
print('seniority mix:')
print(clean['seniority'].value_counts())

---

## 1.9 Scraping Responsibly

A scraper is a program that makes requests to someone else's server, at a speed no human can match, for data they are paying to host. That comes with obligations — some legal, some simply professional.

### The rules

| Rule | Why | How |
|------|-----|-----|
| **Check `robots.txt`** | It states which paths the site asks crawlers to leave alone | Read `site.com/robots.txt` before you start |
| **Read the Terms of Service** | Some sites prohibit automated collection outright | Look for "automated", "crawl", "scrape" |
| **Rate-limit yourself** | A fast loop looks like an attack and can take a small site down | `time.sleep(1)` between requests; never run unthrottled loops |
| **Identify yourself** | Lets an admin contact you instead of blocking you | Set a `User-Agent` with your project name and contact |
| **Cache what you fetch** | Re-parsing a saved file costs the site nothing | Save raw HTML to disk; iterate on the file, not the site |
| **Take only what you need** | Smaller footprint, smaller risk | Don't mirror an entire site to get 10 fields |
| **Avoid personal data** | GDPR, Nigeria's NDPR and similar laws apply to names, emails, phone numbers — scraping does not exempt you | Collect job titles and salaries, not candidate profiles |
| **Never bypass a login or paywall** | Circumventing access controls moves you from "scraping" to something a court may call unauthorised access | If it needs an account you agreed to terms with, respect those terms |

### `robots.txt` in one minute

```
User-agent: *            # these rules apply to every bot
Disallow: /admin/        # please don't fetch anything under /admin/
Crawl-delay: 10          # please wait 10 seconds between requests

User-agent: BadBot       # specific rules for one named bot
Disallow: /              # this bot is asked to fetch nothing at all
```

`robots.txt` is a **request**, not a technical barrier — nothing stops your code from ignoring it. Ignoring it is how you get your IP banned and your employer a letter. Python ships a parser for it in the standard library.

> **The professional position:** scraping public data for analysis is normal and widely done. Hammering a server, collecting personal data, reselling someone's content, or evading access controls is not. Know which side of that line your pipeline is on, and be able to explain it.

In [ ]:
import requests
from urllib.robotparser import RobotFileParser


def robots_check(base_url, path='/', agent='*'):
    '''Report what a site's robots.txt says about fetching `path`.

    We fetch robots.txt with `requests` rather than RobotFileParser.read()
    because requests handles HTTPS certificates reliably on every platform.
    '''
    base = base_url.rstrip('/')
    rp = RobotFileParser()
    try:
        r = requests.get(f'{base}/robots.txt', timeout=10)
    except requests.RequestException as e:
        print(f'{base}  could not fetch robots.txt: {e}')
        return None

    if r.status_code == 200:
        rp.parse(r.text.splitlines())
        note = 'robots.txt found'
    else:
        rp.parse([])                      # no robots.txt = no stated restrictions
        note = f'no robots.txt (HTTP {r.status_code})'

    allowed = rp.can_fetch(agent, base + path)
    delay   = rp.crawl_delay(agent)
    print(f'{base + path:<42} allowed={allowed!s:<5} crawl_delay={delay}   ({note})')
    return allowed


robots_check('https://books.toscrape.com')          # the sandbox we scrape in Lesson 2
robots_check('https://www.python.org', '/')
robots_check('https://www.python.org', '/webstats/')             # disallowed for everyone
robots_check('https://www.python.org', '/', agent='HTTrack')     # this bot is banned outright

---

## Key Takeaways

1. Scraping is the **last resort**. Check for an API, a bulk download, or an open data portal first — a scraper breaks whenever the site's HTML changes.
2. Every scraper is four steps: **fetch → parse → extract → store**. Keep them in separate functions so you can test the parsing without touching the network.
3. HTML is a **tree** of elements with tag names, attributes (`class`, `id`, `href`, `data-*`), and nested children. BeautifulSoup turns that tree into Python objects.
4. Always pass the parser explicitly: `BeautifulSoup(html, 'html.parser')` while learning, `'lxml'` when speed matters.
5. **Find the repeating element first** (`article.job-card`), then extract the same fields from each one. That single habit structures every scraper you will write.
6. `select_one()` returns one match or `None`; `select()` returns a list, empty if nothing matched. `find`/`find_all` do the same with a Python-argument syntax.
7. Learn two selectors properly: **`.class`** and **descendant (a space)**. `.job-card .company` is "a `.company` inside a `.job-card`" — the space is not decoration.
8. Some data lives in **attributes**, not text: `['href']` for links, `['datetime']` for machine-readable dates. Use `.get('attr')` so a missing attribute returns `None` instead of raising `KeyError`.
9. **Never chain off `select_one()`.** Wrap extraction in a helper like `text_at(node, selector, default=None)` — missing fields are the normal case, and one bad record must not end the run.
10. Find selectors with **DevTools → Inspect**, test them with **Ctrl+F in the Elements panel** or `document.querySelectorAll()` in the Console. Write them by hand using meaningful classes; never trust "Copy selector".
11. If a selector works in DevTools but finds nothing in Python, the page is **JavaScript-rendered**. Compare with View Page Source — and see Lesson 3.
12. Scrape responsibly: check `robots.txt` and the ToS, rate-limit every loop, identify your scraper, cache the raw HTML, avoid personal data, and never bypass a login.

---

**Next:** Lesson 2 — fetching live pages with `requests`, handling pagination, and scraping your first real site.